# Build an Intelligent Agent with Semantic Kernel

## Workshop Overview

Welcome to this hands-on tutorial where you'll learn how to build an **intelligent AI agent** using **Microsoft Semantic Kernel**!

This notebook builds upon the RAG concepts from the previous Semantic Kernel tutorial and extends them to create a fully capable AI agent that can:

- 🤖 **Auto-select tools** - Automatically choose which functions to call
- 💭 **Maintain memory** - Remember conversation context and history
- 🔄 **Chain operations** - Combine multiple tools to solve complex tasks
- ⚡ **Stream responses** - Provide real-time feedback during processing
- 🧠 **Reason step-by-step** - Show its thinking process transparently

### What You'll Build

By the end of this tutorial, you'll have created an intelligent agent that can:

1. **Search documents** automatically when questions require specific information
2. **Perform calculations** when mathematical operations are needed
3. **Handle utilities** like text processing, time queries, and formatting
4. **Remember conversations** and build upon previous interactions
5. **Chain multiple tools** together to solve complex multi-step problems

### Agent Architecture in Semantic Kernel

Unlike simple chatbots, **agents** are AI systems that can:
- **Perceive** their environment (through tools and data access)
- **Reason** about what actions to take (via function calling)
- **Act** on the environment (by executing tools and functions)
- **Learn** from interactions (through memory and context)

**Semantic Kernel's agent approach:**
- ✅ **Unified orchestration** - One kernel manages all services and tools
- ✅ **Declarative tools** - Functions decorated with `@kernel_function`
- ✅ **Automatic function calling** - LLM decides which tools to use
- ✅ **Built-in memory** - Conversation state management
- ✅ **Extensible architecture** - Easy to add new capabilities

### Prerequisites

This tutorial assumes you've completed the **Semantic Kernel RAG tutorial**. We'll build upon:
- Kernel setup and Azure OpenAI configuration
- Plugin architecture and function decorators
- Vector embeddings and document search
- Auto function calling capabilities

---

Let's build an intelligent agent! 🚀

## Section 1: Environment Setup and Dependencies

Before building our agent, let's install the required packages and set up our environment.

### What We'll Install:

- **semantic-kernel** - Core AI orchestration SDK
- **python-dotenv** - Environment variable management  
- **aiohttp** - Async HTTP client for web requests
- **numpy** - For numerical operations in tools
- **requests** - For web-based tool functions

### Agent-Specific Features:

Our agent will have enhanced capabilities beyond the basic RAG system:
- **Multiple tool plugins** - Search, calculation, utility functions
- **Conversation memory** - Maintain state across interactions  
- **Streaming responses** - Real-time response generation
- **Multi-step reasoning** - Chain tool calls together

Let's get started!

In [ ]:
# Install required packages for agent development
%pip install semantic-kernel python-dotenv
%pip install aiohttp requests
%pip install numpy
%pip install faiss-cpu

## Section 2: Initialize Semantic Kernel with Azure OpenAI Services

Let's set up our kernel with all the services our agent will need.

### Agent Kernel Architecture

Our agent kernel will include:
1. **Chat Completion Service** - For reasoning and conversation
2. **Text Embedding Service** - For semantic search capabilities
3. **Multiple Plugins** - Tools the agent can use
4. **Memory Management** - Conversation state tracking
5. **Auto Function Calling** - Autonomous tool selection

This creates a complete agent runtime environment!

In [ ]:
import os
from dotenv import load_dotenv
from datetime import datetime
import asyncio

# Import Semantic Kernel components
from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion, AzureTextEmbedding
from semantic_kernel.connectors.ai.function_choice_behavior import FunctionChoiceBehavior
from semantic_kernel.functions import KernelArguments

# Load environment variables
load_dotenv()

# Create the agent kernel
agent_kernel = Kernel()

# Add Azure OpenAI Chat Completion service
chat_completion = AzureChatCompletion(
    service_id="agent_chat",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
)
agent_kernel.add_service(chat_completion)

# Add Azure Text Embedding service for search capabilities
text_embedding = AzureTextEmbedding(
    service_id="agent_embedding",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    deployment_name=os.getenv("AZURE_OPENAI_ADA_DEPLOYMENT")
)
agent_kernel.add_service(text_embedding)

print("✓ Agent Kernel initialized with Azure OpenAI services")
print("✓ Chat completion service added")
print("✓ Text embedding service added")
print("✓ Ready to build agent capabilities!")

## Section 3: Create Tool Functions for the Agent

Now let's build a comprehensive toolkit for our agent! We'll create multiple plugins that give our agent diverse capabilities.

### Agent Tool Architecture

Our agent will have access to several categories of tools:

1. **SearchPlugin** - Web search and information retrieval
2. **CalculatorPlugin** - Mathematical operations and computations  
3. **UtilityPlugin** - Text processing, time queries, and formatting
4. **DocumentPlugin** - Vector search through knowledge base

Each tool is a **@kernel_function** that the agent can automatically choose to use based on the user's request.

### Why Multiple Tools Matter

Real-world agents need diverse capabilities:
- **Search** when users ask about current events or external information
- **Calculate** when mathematical operations are required
- **Process text** when formatting or analysis is needed
- **Query documents** when domain-specific knowledge is required

Let's build these tools!

In [ ]:
import math
import requests
import json
from typing import Union
from semantic_kernel.functions import kernel_function

# 1. Calculator Plugin - Mathematical operations
class CalculatorPlugin:
    """Plugin providing mathematical calculation capabilities for the agent."""
    
    @kernel_function(
        name="calculate",
        description="Performs mathematical calculations. Supports +, -, *, /, **, sqrt, sin, cos, tan, log"
    )
    def calculate(self, expression: str) -> str:
        """Safely evaluates mathematical expressions."""
        try:
            # Replace common math functions
            safe_expr = expression.lower()
            safe_expr = safe_expr.replace("sqrt(", "math.sqrt(")
            safe_expr = safe_expr.replace("sin(", "math.sin(")
            safe_expr = safe_expr.replace("cos(", "math.cos(")
            safe_expr = safe_expr.replace("tan(", "math.tan(")
            safe_expr = safe_expr.replace("log(", "math.log(")
            safe_expr = safe_expr.replace("pi", "math.pi")
            safe_expr = safe_expr.replace("e", "math.e")
            
            # Evaluate safely (restricted namespace)
            allowed_names = {
                "__builtins__": {},
                "math": math,
                "abs": abs,
                "round": round,
                "min": min,
                "max": max
            }
            result = eval(safe_expr, allowed_names)
            return f"Result: {result}"
        except Exception as e:
            return f"Calculation error: {str(e)}"
    
    @kernel_function(
        name="statistics",
        description="Calculates basic statistics (mean, median, mode, std dev) for a list of numbers"
    )
    def statistics(self, numbers: str) -> str:
        """Calculates statistics for comma-separated numbers."""
        try:
            num_list = [float(x.strip()) for x in numbers.split(",")]
            
            mean = sum(num_list) / len(num_list)
            sorted_nums = sorted(num_list)
            n = len(sorted_nums)
            median = sorted_nums[n//2] if n % 2 == 1 else (sorted_nums[n//2-1] + sorted_nums[n//2]) / 2
            
            # Standard deviation
            variance = sum((x - mean) ** 2 for x in num_list) / len(num_list)
            std_dev = math.sqrt(variance)
            
            return f"Statistics: Mean={mean:.2f}, Median={median:.2f}, Std Dev={std_dev:.2f}"
        except Exception as e:
            return f"Statistics error: {str(e)}"


# 2. Utility Plugin - Text processing and general utilities
class UtilityPlugin:
    """Plugin providing text processing and utility functions for the agent."""
    
    @kernel_function(
        name="get_current_time",
        description="Returns the current date and time"
    )
    def get_current_time(self) -> str:
        """Gets current timestamp."""
        now = datetime.now()
        return f"Current time: {now.strftime('%Y-%m-%d %H:%M:%S')}"
    
    @kernel_function(
        name="text_analysis",
        description="Analyzes text for word count, character count, and reading time estimate"
    )
    def text_analysis(self, text: str) -> str:
        """Analyzes text properties."""
        words = len(text.split())
        characters = len(text)
        characters_no_spaces = len(text.replace(" ", ""))
        sentences = len([s for s in text.split(".") if s.strip()])
        
        # Estimate reading time (average 200 words per minute)
        reading_time = words / 200
        
        return f"Text Analysis: {words} words, {characters} chars ({characters_no_spaces} no spaces), {sentences} sentences, ~{reading_time:.1f} min read time"
    
    @kernel_function(
        name="format_text",
        description="Formats text in various ways: uppercase, lowercase, title case, or sentence case"
    )
    def format_text(self, text: str, format_type: str = "title") -> str:
        """Formats text according to specified type."""
        format_type = format_type.lower()
        
        if format_type == "upper":
            return text.upper()
        elif format_type == "lower":
            return text.lower()
        elif format_type == "title":
            return text.title()
        elif format_type == "sentence":
            return text.capitalize()
        else:
            return f"Unknown format '{format_type}'. Use: upper, lower, title, or sentence"


# 3. Search Plugin - Web search capabilities
class SearchPlugin:
    """Plugin providing web search capabilities for the agent."""
    
    @kernel_function(
        name="web_search",
        description="Searches the web for current information about a topic"
    )
    def web_search(self, query: str) -> str:
        """Simulated web search function (replace with real API in production)."""
        # In a real implementation, you'd use APIs like Bing Search, Google Custom Search, etc.
        search_results = {
            "AI": "AI (Artificial Intelligence) refers to computer systems that can perform tasks typically requiring human intelligence. Recent developments include large language models like GPT-4, multimodal AI systems, and advances in computer vision.",
            "machine learning": "Machine learning is a subset of AI focusing on algorithms that improve through experience. Current trends include transformer architectures, reinforcement learning, and neural network optimization techniques.",
            "semantic kernel": "Semantic Kernel is Microsoft's SDK for integrating AI services into applications. It provides a plugin architecture for combining AI models, prompts, and native code into unified workflows.",
            "weather": "Weather information varies by location and time. For accurate weather data, check local meteorological services or weather APIs like OpenWeatherMap.",
            "news": "Current news covers various topics including technology, politics, science, and world events. For up-to-date news, consult reliable news sources and fact-checking services."
        }
        
        # Find best match
        query_lower = query.lower()
        for keyword, info in search_results.items():
            if keyword in query_lower:
                return f"Search results for '{query}': {info}"
        
        return f"Search results for '{query}': No specific information found in demo database. In production, this would query real search APIs."
    
    @kernel_function(
        name="get_definition",
        description="Gets the definition of a term or concept"
    )
    def get_definition(self, term: str) -> str:
        """Gets definition of a term."""
        definitions = {
            "agent": "An AI agent is an autonomous system that perceives its environment, makes decisions, and takes actions to achieve specific goals.",
            "kernel": "In Semantic Kernel, a kernel is the central orchestrator that manages AI services, plugins, and functions.",
            "plugin": "A plugin in Semantic Kernel is a collection of functions that extend the kernel's capabilities.",
            "embedding": "An embedding is a vector representation of text that captures semantic meaning for similarity search.",
            "rag": "RAG (Retrieval Augmented Generation) combines retrieval from knowledge sources with generative AI to provide accurate, grounded responses.",
            "function calling": "Function calling enables AI models to automatically select and execute appropriate tools based on user requests."
        }
        
        term_lower = term.lower()
        if term_lower in definitions:
            return f"Definition of '{term}': {definitions[term_lower]}"
        else:
            return f"Definition not found for '{term}'. Try searching for more information."

# Add plugins to the agent kernel
agent_kernel.add_plugin(CalculatorPlugin(), plugin_name="Calculator")
agent_kernel.add_plugin(UtilityPlugin(), plugin_name="Utility")
agent_kernel.add_plugin(SearchPlugin(), plugin_name="Search")

print("✓ Calculator Plugin added - Mathematical operations and statistics")
print("✓ Utility Plugin added - Text processing and time functions")
print("✓ Search Plugin added - Web search and definitions")
print("✓ Agent now has access to diverse tool capabilities!")

## Section 4: Set up Vector Store and Document Collection

Now let's add document search capabilities to our agent by setting up vector storage with embedded knowledge.

### Agent Knowledge Base

Our agent will have access to a curated knowledge base about AI and technology topics. This enables the agent to:

- **Answer domain-specific questions** with accurate, source-grounded information
- **Distinguish between search and retrieval** - use web search for current events, document search for specific knowledge
- **Combine multiple information sources** - query documents and then perform calculations on the results

### Integration with Existing Tools

The document search capability works seamlessly with other tools:
- Search documents for AI concepts, then use calculator for related metrics
- Retrieve definitions from knowledge base, then format text appropriately
- Chain document retrieval with web search for comprehensive answers

Let's set up the vector store!

In [ ]:
from dataclasses import dataclass
from typing import Annotated
from semantic_kernel.data.vector import VectorStoreField, vectorstoremodel
from semantic_kernel.connectors.in_memory import InMemoryCollection

# Define document model with vector store decorator
@vectorstoremodel(collection_name="agent_documents")
@dataclass
class AgentDocument:
    """Document model for agent's knowledge base."""
    id: Annotated[str, VectorStoreField("key")]
    text: Annotated[str, VectorStoreField("data")]
    topic: Annotated[str, VectorStoreField("data")]  # Topic categorization
    embedding: Annotated[
        list[float] | None,
        VectorStoreField(
            "vector",
            dimensions=1536,
            embedding_generator=text_embedding
        ),
    ] = None

# Enhanced knowledge base with more diverse content for the agent
AGENT_KNOWLEDGE_BASE = """
Artificial Intelligence Agents are autonomous systems that can perceive their environment, make decisions, and take actions to achieve specific goals. Unlike traditional software, agents exhibit adaptive behavior and can learn from interactions. Modern AI agents combine large language models with tool access, enabling them to perform complex multi-step tasks like web search, data analysis, and workflow automation.

Semantic Kernel Architecture provides a unified framework for building AI applications. The kernel acts as a central orchestrator managing multiple services including chat completion, embeddings, and memory. Plugins extend functionality through decorated functions, while the execution engine handles automatic function calling and prompt orchestration. This architecture enables developers to build sophisticated AI agents with minimal boilerplate code.

Function Calling in AI Systems enables models to automatically select and execute appropriate tools based on user requests. This capability transforms static chatbots into dynamic agents capable of real-world task completion. The process involves the model analyzing user intent, selecting relevant functions from available tools, executing those functions, and integrating results into coherent responses.

Vector Databases and Embeddings enable semantic search by converting text into high-dimensional numerical representations. Similar concepts cluster together in vector space, allowing for meaning-based rather than keyword-based search. This technology powers RAG systems, recommendation engines, and content discovery platforms by finding semantically related information even when exact keywords don't match.

Machine Learning Operations (MLOps) encompasses the practices and tools for deploying, monitoring, and maintaining machine learning models in production. Key components include model versioning, automated testing, performance monitoring, and deployment pipelines. Effective MLOps ensures reliable, scalable, and maintainable AI systems that can adapt to changing requirements and data distributions.

Natural Language Processing has evolved from rule-based systems to transformer architectures that achieve human-level performance on many tasks. Modern NLP combines statistical learning with neural networks to understand context, sentiment, and intent. Applications include translation, summarization, question answering, and conversational AI systems that can engage in complex multi-turn dialogues.

Retrieval Augmented Generation (RAG) combines the knowledge stored in large language models with external information retrieval. This hybrid approach enables AI systems to access up-to-date information while maintaining the reasoning capabilities of foundation models. RAG systems typically involve document embedding, similarity search, and prompt augmentation to provide accurate, grounded responses.

Transformer Architecture revolutionized deep learning through self-attention mechanisms that process sequences in parallel rather than sequentially. This innovation enables efficient training on large datasets and captures long-range dependencies in text, code, and other structured data. Transformers form the foundation of modern language models including GPT, BERT, and their successors.

Agent Memory and State Management are crucial for maintaining context across multi-turn conversations and complex task execution. Memory systems can be episodic (remembering specific interactions), semantic (storing general knowledge), or procedural (maintaining workflow state). Effective memory management enables agents to build upon previous interactions and maintain coherent long-term behavior.

Prompt Engineering involves designing inputs to language models that elicit desired behaviors and outputs. Techniques include few-shot learning, chain-of-thought reasoning, role-playing, and structured templates. Advanced prompt engineering can significantly improve model performance on specific tasks without requiring model fine-tuning or retraining.
"""

# Split knowledge base into documents by topic
def create_agent_documents(text: str) -> list[AgentDocument]:
    """Create document objects from knowledge base text."""
    paragraphs = [p.strip() for p in text.strip().split('\n\n') if p.strip()]
    documents = []
    
    for i, paragraph in enumerate(paragraphs):
        # Extract topic from first few words
        topic = paragraph.split()[0:2]
        topic_str = " ".join(topic)
        
        documents.append(AgentDocument(
            id=f"doc_{i}",
            text=paragraph,
            topic=topic_str
        ))
    
    return documents

# Create document collection
agent_documents = create_agent_documents(AGENT_KNOWLEDGE_BASE)

# Set up vector collection
document_collection = InMemoryCollection(record_type=AgentDocument)
await document_collection.ensure_collection_exists()

# Index documents with embeddings
await document_collection.upsert(agent_documents)

# Create search function and add to kernel
agent_kernel.add_function(
    "Knowledge",
    document_collection.create_search_function(
        function_name="search_knowledge",
        description="Searches the agent's knowledge base for information about AI, ML, and technology topics. Use this for domain-specific questions about artificial intelligence, machine learning, and software development.",
        string_mapper=lambda x: f"Topic: {x.record.topic}\nContent: {x.record.text}",
    ),
)

print(f"✓ Knowledge base created with {len(agent_documents)} documents")
print("✓ Vector embeddings generated for all documents")
print("✓ Document search function 'Knowledge.search_knowledge' added to agent")
print("✓ Agent can now search its knowledge base automatically!")

## Section 5: Build Agent with Auto Function Calling

Now for the magic! Let's configure our agent to automatically select and call the appropriate tools based on user requests.

### Auto Function Calling Architecture

With `FunctionChoiceBehavior.Auto()`, our agent will:

1. **Analyze user intent** - Understand what the user is asking for
2. **Select appropriate tools** - Choose from Calculator, Utility, Search, or Knowledge functions
3. **Execute functions** - Call the selected tools with proper parameters
4. **Synthesize results** - Combine tool outputs into coherent responses
5. **Chain operations** - Use multiple tools sequentially when needed

### Agent Intelligence Levels

Our agent can handle increasingly complex scenarios:

- **Simple queries** - Direct answers without tool usage
- **Single tool tasks** - Calculate math, search web, format text
- **Multi-tool workflows** - Search knowledge base, then perform calculations
- **Reasoning chains** - Combine information from multiple sources

### Creating the Agent Persona

We'll give our agent a helpful, knowledgeable personality that:
- Explains its tool usage transparently
- Shows step-by-step reasoning  
- Provides detailed, accurate responses
- Adapts to user needs and context

Let's bring our agent to life!

In [ ]:
# Configure auto function calling for the agent
agent_execution_settings = agent_kernel.get_prompt_execution_settings_from_service_id("agent_chat")
agent_execution_settings.function_choice_behavior = FunctionChoiceBehavior.Auto()

# Define the agent's personality and capabilities
AGENT_SYSTEM_PROMPT = """
You are SKAgent, an intelligent AI assistant powered by Microsoft Semantic Kernel. You have access to multiple tools and capabilities that you can use automatically to help users.

## Your Available Tools:
- **Calculator**: Perform mathematical calculations, statistics, and numerical analysis
- **Utility**: Get current time, analyze text, format text in various ways  
- **Search**: Search the web for current information and get definitions
- **Knowledge**: Search your knowledge base for information about AI, ML, and technology topics

## Your Behavior Guidelines:
1. **Be proactive**: Automatically use tools when they would be helpful
2. **Be transparent**: Explain which tools you're using and why
3. **Be thorough**: Provide comprehensive answers with relevant details
4. **Be intelligent**: Chain multiple tools together when needed
5. **Be helpful**: Always aim to fully address the user's needs

## When to Use Each Tool:
- Use **Calculator** for any mathematical operations, statistics, or numerical analysis
- Use **Utility** for text processing, formatting, or getting current time
- Use **Search** for current events, general web information, or definitions of common terms
- Use **Knowledge** for detailed information about AI, machine learning, Semantic Kernel, or technical topics

## Multi-Tool Workflows:
You can combine tools creatively:
- Search knowledge base for technical info, then calculate related metrics
- Get definitions, then format the response appropriately
- Retrieve information from multiple sources and synthesize comprehensive answers

Always explain your reasoning and show your work when using tools.
"""

# Create the main agent function
async def run_agent(user_message: str, show_reasoning: bool = True) -> str:
    """
    Runs the agent with auto function calling enabled.
    
    Args:
        user_message: The user's question or request
        show_reasoning: Whether to show the agent's reasoning process
    """
    
    # Create the full prompt with system instructions
    full_prompt = f"""
    {AGENT_SYSTEM_PROMPT}
    
    User Message: {user_message}
    
    Please help the user with their request. Use your available tools as needed and provide a comprehensive response.
    """
    
    try:
        # Execute with auto function calling
        result = await agent_kernel.invoke_prompt(
            prompt=full_prompt,
            function_name="agent_response",
            plugin_name="SKAgent",
            arguments=KernelArguments(settings=agent_execution_settings)
        )
        
        return str(result)
        
    except Exception as e:
        return f"Error: {str(e)}"

# Test the agent setup
print("✅ Agent configured with auto function calling!")
print("✅ System prompt defined with tool usage guidelines")
print("✅ Agent function created and ready to use")
print("\nAgent capabilities:")
print("🧮 Calculator - Mathematical operations and statistics")
print("🔧 Utility - Text processing and time functions")
print("🌐 Search - Web search and definitions")
print("📚 Knowledge - AI/ML knowledge base search")
print("\n🤖 SKAgent is ready to help!")

## Section 6: Add Conversation Memory and State Management  

Let's enhance our agent with memory capabilities to maintain context across multiple interactions!

### Agent Memory Architecture

Our enhanced agent will include:

1. **Conversation History** - Remember previous messages and responses
2. **Context Awareness** - Build upon earlier interactions  
3. **Session Management** - Maintain state within conversation sessions
4. **Persistent Context** - Reference earlier calculations, searches, or information

### Why Memory Matters for Agents

Memory transforms our agent from a stateless responder to an intelligent conversational partner:

- **Continuity** - "Remember what we calculated earlier"
- **Personalization** - Learn user preferences and context
- **Efficiency** - Avoid repeating searches or calculations
- **Complex workflows** - Build multi-step processes over time

Let's implement conversation memory!

In [ ]:
from typing import List, Dict
from dataclasses import dataclass
from datetime import datetime

@dataclass
class ConversationMessage:
    """Represents a single message in the conversation."""
    role: str  # "user" or "assistant"
    content: str
    timestamp: datetime
    tools_used: List[str] = None  # Track which tools were used

class AgentMemory:
    """Manages conversation memory and state for the agent."""
    
    def __init__(self):
        self.conversation_history: List[ConversationMessage] = []
        self.session_context: Dict[str, any] = {}
        self.tool_results_cache: Dict[str, any] = {}
    
    def add_message(self, role: str, content: str, tools_used: List[str] = None):
        """Add a message to conversation history."""
        message = ConversationMessage(
            role=role,
            content=content,
            timestamp=datetime.now(),
            tools_used=tools_used or []
        )
        self.conversation_history.append(message)
    
    def get_conversation_summary(self, last_n: int = 5) -> str:
        """Get a summary of recent conversation."""
        if not self.conversation_history:
            return "No previous conversation history."
        
        recent_messages = self.conversation_history[-last_n:]
        summary = "## Recent Conversation History:\\n"
        
        for msg in recent_messages:
            role_icon = "👤" if msg.role == "user" else "🤖"
            tools_info = f" (Used: {', '.join(msg.tools_used)})" if msg.tools_used else ""
            summary += f"{role_icon} **{msg.role.title()}**: {msg.content[:100]}{'...' if len(msg.content) > 100 else ''}{tools_info}\\n"
        
        return summary
    
    def set_context(self, key: str, value: any):
        """Store context information for the session."""
        self.session_context[key] = value
    
    def get_context(self, key: str, default=None):
        """Retrieve context information."""
        return self.session_context.get(key, default)
    
    def cache_tool_result(self, tool_call: str, result: str):
        """Cache tool results to avoid repeated calls."""
        self.tool_results_cache[tool_call] = {
            'result': result,
            'timestamp': datetime.now()
        }
    
    def get_cached_result(self, tool_call: str) -> str:
        """Retrieve cached tool result if available and recent."""
        cached = self.tool_results_cache.get(tool_call)
        if cached:
            # Consider cache valid for 5 minutes
            time_diff = datetime.now() - cached['timestamp']
            if time_diff.seconds < 300:  # 5 minutes
                return cached['result']
        return None

# Initialize agent memory
agent_memory = AgentMemory()

# Enhanced agent function with memory
async def run_agent_with_memory(user_message: str, show_reasoning: bool = True) -> str:
    """
    Runs the agent with memory and conversation context.
    """
    
    # Add user message to memory
    agent_memory.add_message("user", user_message)
    
    # Get conversation context
    conversation_context = agent_memory.get_conversation_summary()
    
    # Create enhanced prompt with memory context
    memory_enhanced_prompt = f"""
    {AGENT_SYSTEM_PROMPT}
    
    {conversation_context}
    
    ## Current User Request: {user_message}
    
    Please help the user with their request. Consider the conversation history and use your available tools as needed. 
    If the user refers to "earlier" calculations or information, check the conversation history above.
    Provide a comprehensive response and remember this interaction for future reference.
    """
    
    try:
        # Execute with auto function calling
        result = await agent_kernel.invoke_prompt(
            prompt=memory_enhanced_prompt,
            function_name="agent_memory_response",
            plugin_name="SKAgent",
            arguments=KernelArguments(settings=agent_execution_settings)
        )
        
        response = str(result)
        
        # Add agent response to memory
        # Note: In a real implementation, you'd track which tools were actually used
        agent_memory.add_message("assistant", response, ["auto-detected"])
        
        return response
        
    except Exception as e:
        error_response = f"Error: {str(e)}"
        agent_memory.add_message("assistant", error_response)
        return error_response

# Conversation management functions
def start_new_conversation():
    """Start a new conversation session."""
    global agent_memory
    agent_memory = AgentMemory()
    print("🔄 New conversation started. Memory cleared.")

def show_conversation_history():
    """Display the current conversation history."""
    if not agent_memory.conversation_history:
        print("📝 No conversation history yet.")
        return
    
    print("📜 **Conversation History:**")
    for i, msg in enumerate(agent_memory.conversation_history, 1):
        role_icon = "👤" if msg.role == "user" else "🤖"
        tools_info = f" [Tools: {', '.join(msg.tools_used)}]" if msg.tools_used else ""
        print(f"{i}. {role_icon} **{msg.role.title()}** ({msg.timestamp.strftime('%H:%M:%S')}){tools_info}")
        print(f"   {msg.content[:200]}{'...' if len(msg.content) > 200 else ''}")
        print()

print("✅ Agent memory system initialized!")
print("✅ Conversation history tracking enabled")
print("✅ Session context management ready")
print("✅ Tool result caching implemented")
print("\n🧠 Agent now has memory and can maintain conversation context!")

## Section 7: Test Agent with Tool Usage

Time to put our agent to the test! Let's see how it automatically selects and uses different tools.

### Test Scenarios

We'll test our agent with different types of queries to demonstrate:

1. **Calculator Usage** - Mathematical problems and statistics
2. **Knowledge Base Search** - AI and technology questions  
3. **Utility Functions** - Text processing and time queries
4. **Web Search** - Current information and definitions
5. **Multi-tool Workflows** - Complex tasks requiring multiple tools

### Agent Intelligence Demo

Watch how the agent:
- **Analyzes** the user's intent
- **Selects** appropriate tools automatically
- **Executes** functions with proper parameters
- **Synthesizes** coherent responses
- **Explains** its reasoning process

Let's see our agent in action!

In [ ]:
# Test 1: Calculator Tool Usage
print("🧮 **Test 1: Calculator Tool Usage**")
print("="*50)

test_math = await run_agent_with_memory(
    "Can you calculate the square root of 144 and then find the mean of the numbers 10, 15, 20, 25, 30?"
)
print(f"User: Can you calculate the square root of 144 and then find the mean of the numbers 10, 15, 20, 25, 30?")
print(f"Agent: {test_math}")
print()

In [ ]:
# Test 2: Knowledge Base Search
print("📚 **Test 2: Knowledge Base Search**")
print("="*50)

test_knowledge = await run_agent_with_memory(
    "What is RAG and how does it work with embeddings?"
)
print(f"User: What is RAG and how does it work with embeddings?")
print(f"Agent: {test_knowledge}")
print()

In [ ]:
# Test 3: Utility Functions
print("🔧 **Test 3: Utility Functions**")
print("="*50)

test_utility = await run_agent_with_memory(
    "What's the current time and can you analyze this text: 'Semantic Kernel is a powerful framework for building AI applications with function calling capabilities'?"
)
print(f"User: What's the current time and can you analyze this text: 'Semantic Kernel is a powerful framework for building AI applications with function calling capabilities'?")
print(f"Agent: {test_utility}")
print()

In [ ]:
# Test 4: Web Search
print("🌐 **Test 4: Web Search**")
print("="*50)

test_search = await run_agent_with_memory(
    "Can you search for information about machine learning and give me a definition of 'agent'?"
)
print(f"User: Can you search for information about machine learning and give me a definition of 'agent'?")
print(f"Agent: {test_search}")
print()

In [ ]:
# Test 5: Memory and Context
print("🧠 **Test 5: Memory and Context**")
print("="*50)

test_memory = await run_agent_with_memory(
    "Based on our earlier conversation, can you remind me what we calculated and then format it as a title?"
)
print(f"User: Based on our earlier conversation, can you remind me what we calculated and then format it as a title?")
print(f"Agent: {test_memory}")
print()

# Show conversation history
print("📜 **Conversation History So Far:**")
show_conversation_history()

## Section 8: Implement Streaming Agent Responses

Let's add streaming capabilities to provide real-time responses as our agent works!

### Why Streaming Matters for Agents

Streaming enhances the user experience by:

- **Real-time feedback** - Users see responses as they're generated
- **Progress indication** - Show when tools are being called
- **Transparency** - Reveal the agent's thinking process step-by-step  
- **Engagement** - Keep users engaged during longer operations

### Streaming Architecture

Our streaming agent will:
1. **Stream text generation** - Show response as it's being created
2. **Indicate tool calls** - Show when functions are being executed
3. **Display intermediate results** - Show tool outputs as they complete
4. **Provide status updates** - Keep users informed of progress

This creates a more interactive and transparent agent experience!

In [ ]:
import asyncio
import sys
from typing import AsyncGenerator

class StreamingAgent:
    """Enhanced agent with streaming capabilities."""
    
    def __init__(self, kernel: Kernel, memory: AgentMemory):
        self.kernel = kernel
        self.memory = memory
        self.execution_settings = kernel.get_prompt_execution_settings_from_service_id("agent_chat")
        self.execution_settings.function_choice_behavior = FunctionChoiceBehavior.Auto()
    
    async def stream_response(self, user_message: str) -> AsyncGenerator[str, None]:
        """
        Stream agent response with real-time updates.
        """
        # Add user message to memory
        self.memory.add_message("user", user_message)
        
        # Status update
        yield "🤖 **SKAgent is thinking...**\n\n"
        await asyncio.sleep(0.5)  # Brief pause for effect
        
        # Get conversation context
        conversation_context = self.memory.get_conversation_summary()
        
        # Create enhanced prompt
        memory_enhanced_prompt = f"""
        {AGENT_SYSTEM_PROMPT}
        
        {conversation_context}
        
        ## Current User Request: {user_message}
        
        Please help the user with their request. Consider the conversation history and use your available tools as needed.
        """
        
        try:
            yield "🔍 **Analyzing request and selecting tools...**\n\n"
            await asyncio.sleep(0.5)
            
            # Execute with function calling (in real implementation, you'd hook into the streaming)
            result = await self.kernel.invoke_prompt(
                prompt=memory_enhanced_prompt,
                function_name="streaming_agent_response",
                plugin_name="SKAgent",
                arguments=KernelArguments(settings=self.execution_settings)
            )
            
            # Simulate streaming the response
            response_text = str(result)
            
            yield "✅ **Tools executed, generating response:**\n\n"
            await asyncio.sleep(0.3)
            
            # Stream the response word by word for demo
            words = response_text.split()
            current_text = ""
            
            for i, word in enumerate(words):
                current_text += word + " "
                
                # Stream every few words
                if i % 3 == 0 or i == len(words) - 1:
                    yield current_text[len(current_text) - len(word) - 1:]
                    await asyncio.sleep(0.1)
            
            # Add to memory
            self.memory.add_message("assistant", response_text, ["auto-detected"])
            
        except Exception as e:
            error_msg = f"\n\n❌ **Error:** {str(e)}"
            yield error_msg
            self.memory.add_message("assistant", error_msg)

# Create streaming agent instance
streaming_agent = StreamingAgent(agent_kernel, agent_memory)

async def demo_streaming_response(query: str):
    """Demonstrate streaming agent response."""
    print(f"👤 **User:** {query}\n")
    
    complete_response = ""
    async for chunk in streaming_agent.stream_response(query):
        print(chunk, end="", flush=True)
        complete_response += chunk
        
    print("\n" + "="*60 + "\n")
    return complete_response

print("✅ Streaming agent implementation ready!")
print("✅ Real-time response generation enabled")
print("✅ Progressive tool execution feedback")
print("\n⚡ Ready to demonstrate streaming capabilities!")

In [ ]:
# Demo streaming responses
print("🌊 **Streaming Agent Demo**")
print("="*50)

# Test streaming with a complex query that requires multiple tools
streaming_response = await demo_streaming_response(
    "Calculate the area of a circle with radius 7, then search your knowledge base for information about transformers, and format the result nicely"
)

## Section 9: Add Multi-Tool Agent Workflows

Let's create complex scenarios where our agent chains multiple tools together to solve sophisticated problems!

### Advanced Agent Workflows

Our agent can now handle complex multi-step processes:

1. **Sequential Tool Chaining** - Use results from one tool as input to another
2. **Conditional Logic** - Choose different tools based on intermediate results
3. **Data Processing Pipelines** - Search → Calculate → Format → Present
4. **Contextual Reasoning** - Build complex answers from multiple information sources

### Real-World Scenarios

These advanced workflows enable our agent to handle tasks like:

- **Research & Analysis** - Search for data, perform calculations, format results
- **Content Generation** - Retrieve information, process text, create summaries
- **Problem Solving** - Break down complex problems into tool-solvable steps
- **Decision Support** - Gather data from multiple sources and provide recommendations

### Multi-Tool Intelligence

Watch how our agent demonstrates true intelligence by:
- **Planning** multi-step approaches to complex problems
- **Adapting** its strategy based on intermediate results  
- **Combining** information from diverse sources
- **Reasoning** through complex logical sequences

Let's test these advanced capabilities!

In [ ]:
# Complex Multi-Tool Workflow Test 1: Research and Analysis
print("🔗 **Multi-Tool Workflow 1: Research and Analysis**")
print("="*60)

workflow_1 = await run_agent_with_memory(
    "I need to understand Semantic Kernel agents. First, search your knowledge base for information about AI agents, then calculate how many words are in a typical agent description, and finally format the key points in title case."
)
print(f"User: I need to understand Semantic Kernel agents. First, search your knowledge base for information about AI agents, then calculate how many words are in a typical agent description, and finally format the key points in title case.")
print(f"Agent: {workflow_1}")
print()

In [ ]:
# Complex Multi-Tool Workflow Test 2: Data Processing Pipeline
print("📊 **Multi-Tool Workflow 2: Data Processing Pipeline**")
print("="*60)

workflow_2 = await run_agent_with_memory(
    "Let's do some analysis: First get the current time, then calculate statistics for these performance scores: 85, 92, 78, 96, 88, 91, 84. Finally, search for information about machine learning performance metrics in your knowledge base."
)
print(f"User: Let's do some analysis: First get the current time, then calculate statistics for these performance scores: 85, 92, 78, 96, 88, 91, 84. Finally, search for information about machine learning performance metrics in your knowledge base.")
print(f"Agent: {workflow_2}")
print()

In [ ]:
# Final workflow test with memory reference
print("🧠 **Multi-Tool Workflow 3: Memory-Based Decision Making**")
print("="*60)

workflow_3 = await run_agent_with_memory(
    "Based on all our previous conversations and calculations, can you summarize what we've learned about agent capabilities, include the statistics we calculated earlier, and search for information about how this relates to real-world AI applications?"
)
print(f"User: Based on all our previous conversations and calculations, can you summarize what we've learned about agent capabilities, include the statistics we calculated earlier, and search for information about how this relates to real-world AI applications?")
print(f"Agent: {workflow_3}")
print()

print("🎯 **Multi-Tool Workflow Analysis Complete!**")
print("="*60)
print("✅ Agent successfully demonstrated:")
print("   • Sequential tool chaining")
print("   • Memory-based context awareness") 
print("   • Complex reasoning workflows")
print("   • Information synthesis across tools")
print("   • Adaptive response generation")

---

## 🎓 Conclusion: Building Intelligent Agents with Semantic Kernel

Congratulations! You've successfully built a complete intelligent agent using Microsoft Semantic Kernel. This tutorial has taken you from basic tool creation to advanced multi-tool workflows and memory management.

### 🏗️ What You've Built

Your **SKAgent** now includes:

| Component | Capability | Implementation |
|-----------|------------|----------------|
| **🧮 Calculator Plugin** | Mathematical operations & statistics | `@kernel_function` decorators |
| **🔧 Utility Plugin** | Text processing & time functions | Native Python functions |
| **🌐 Search Plugin** | Web search & definitions | Simulated search API |
| **📚 Knowledge Plugin** | Vector-based document search | InMemoryCollection + embeddings |
| **🧠 Memory System** | Conversation context & history | Custom AgentMemory class |
| **⚡ Streaming** | Real-time response generation | AsyncGenerator streaming |
| **🔗 Multi-Tool Workflows** | Complex reasoning chains | Auto function calling |

### 🚀 Agent Capabilities Demonstrated

Throughout this tutorial, your agent has shown:

1. **Autonomous Tool Selection** - Automatically choosing appropriate functions
2. **Context Awareness** - Remembering and building upon previous interactions
3. **Multi-Step Reasoning** - Chaining tools together for complex tasks
4. **Real-Time Feedback** - Streaming responses as they're generated
5. **Intelligent Synthesis** - Combining information from multiple sources

### 🔑 Key Semantic Kernel Concepts Mastered

| Concept | Purpose | Agent Benefit |
|---------|---------|---------------|
| **Kernel** | Central orchestrator | Unified service management |
| **@kernel_function** | Tool registration | Automatic function discovery |
| **FunctionChoiceBehavior.Auto()** | Autonomous tool selection | Intelligent decision making |
| **InMemoryCollection** | Vector storage | Semantic knowledge search |
| **Prompt Templates** | Structured prompting | Consistent agent behavior |
| **Memory Management** | State persistence | Conversational continuity |

### 🆚 Agent vs. Chatbot Comparison

**Traditional Chatbot:**
- ❌ Static responses from training data
- ❌ No tool access or external capabilities
- ❌ No memory between conversations
- ❌ Limited to text generation only

**Your Semantic Kernel Agent:**
- ✅ **Dynamic tool usage** - Can perform calculations, searches, data processing
- ✅ **External knowledge access** - Retrieves up-to-date information  
- ✅ **Persistent memory** - Remembers conversation context
- ✅ **Multi-modal capabilities** - Text, calculations, search, analysis
- ✅ **Reasoning chains** - Combines multiple tools intelligently

### 🛠️ Production Considerations

To deploy your agent in production, consider:

1. **Security**
   - Validate all tool inputs
   - Implement rate limiting
   - Use secure API key management

2. **Scalability**
   - Replace InMemoryCollection with production vector DB (Azure AI Search, Qdrant)
   - Implement persistent memory storage
   - Add load balancing for high traffic

3. **Monitoring**
   - Track tool usage and performance
   - Log conversation flows
   - Monitor costs and API usage

4. **Enhanced Tools**
   - Integrate real web search APIs (Bing, Google)
   - Add database connectivity
   - Include file processing capabilities
   - Connect to business systems

### 🔮 Advanced Extensions

Ready for more? Extend your agent with:

1. **Specialized Domains**
   ```python
   class DatabasePlugin:
       @kernel_function
       def query_database(self, sql: str) -> str:
           # Execute SQL queries
   ```

2. **File Processing**
   ```python
   class FilePlugin:
       @kernel_function
       def process_document(self, file_path: str) -> str:
           # Read and analyze documents
   ```

3. **API Integrations**
   ```python
   class APIPlugin:
       @kernel_function
       def call_rest_api(self, endpoint: str, data: str) -> str:
           # Make REST API calls
   ```

4. **Multi-Agent Coordination**
   - Create specialized agents for different domains
   - Implement agent-to-agent communication
   - Build hierarchical agent systems

### 📚 Next Steps & Resources

**Continue Learning:**
- [Semantic Kernel Documentation](https://learn.microsoft.com/semantic-kernel/)
- [Azure OpenAI Service](https://learn.microsoft.com/azure/ai-services/openai/)
- [Semantic Kernel Samples](https://github.com/microsoft/semantic-kernel/tree/main/python/samples)

**Build Production Agents:**
- Implement proper error handling and retries
- Add comprehensive logging and monitoring
- Create agent configuration management
- Build user authentication and authorization
- Design multi-tenant agent architectures

**Explore Advanced Topics:**
- **Planners** - Multi-step reasoning and planning
- **Memory Connectors** - Persistent storage solutions
- **Custom Connectors** - Integrate any LLM or service
- **Agent Orchestration** - Coordinate multiple specialized agents

---

### 🙏 Thank You!

You now have the knowledge and tools to build production-ready intelligent agents with Microsoft Semantic Kernel. Your agent demonstrates the full spectrum of capabilities - from simple tool usage to complex multi-step reasoning with memory.

**Key Takeaways:**
- ✅ Semantic Kernel provides a unified framework for agent development
- ✅ The plugin architecture makes tools modular and reusable
- ✅ Auto function calling enables true agent intelligence
- ✅ Memory management creates conversational continuity
- ✅ Multi-tool workflows solve complex real-world problems

**Keep Building! 🚀**

The future of AI is agentic - systems that can perceive, reason, and act autonomously. With Semantic Kernel, you're well-equipped to build the next generation of intelligent applications.

Happy coding! 🎉

In [ ]:
# Start a fresh conversation for complex workflow tests
start_new_conversation()

print("🔄 **Multi-Tool Workflow Tests**")
print("="*60)

# Test 1: Research, Calculate, and Format Workflow
print("\n🔬 **Test 1: Research → Calculate → Format Workflow**")
print("-" * 50)

workflow_test_1 = await run_agent_with_memory(
    """I need help with a machine learning project analysis. Can you:
    1. Search your knowledge base for information about transformer architectures
    2. Calculate the total parameters if a transformer has 12 layers, each with 768 hidden units, and vocabulary size of 30,000
    3. Format the final analysis as a nice summary
    
    Please walk me through each step of your analysis."""
)

print(f"🤖 Agent Response:\n{workflow_test_1}")
print("\n" + "="*60)